# Clase 3 · 03 · Documental: un cliente, un documento

En una base documental (MongoDB, Elasticsearch) la unidad es el **documento**: un objeto JSON con campos anidados y listas. Se diseña a partir de **cómo se lee el dato**: lo que se consulta junto, se guarda junto.

Vamos a convertir las tres tablas relacionales en una colección de documentos de cliente:

```json
{
  "_id": 42,
  "profile": {"country": "AR", "segment": "premium", "email": "..."},
  "stats": {"tx_count": 11, "total_amount": 9876.5, "fraud_transactions": 1},
  "transactions": [
    {"event_ts": "...", "transaction_id": 123, "amount": 450.2, "payment_channel": "card",
     "device_id": "device_17", "product": {"product_id": 8, "category": "books", "price": 30.0}}
  ]
}
```

Las transacciones quedan **embebidas** en el cliente y el producto se copia como **snapshot** dentro de cada transacción. El `product_id` se conserva como **referencia** al catálogo.

In [ ]:
dbutils.widgets.text("student_id", "alumno01", "01 - Identificador")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "02 - Escala")

In [ ]:
%run ../common/config

In [ ]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

catalog = spark.sql("SELECT current_catalog()").first()[0]
config = CourseConfig(catalog, dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)

customers = qualified_table(config, "silver_customers")
products = qualified_table(config, "silver_products")
transactions = qualified_table(config, "silver_transactions")
for table in [customers, products, transactions]:
    assert spark.catalog.tableExists(table), f"Falta {table}. Ejecutá 00_preflight (y la clase 2)."
lab_path = f"{config.volume_path}/nosql_lab"
print(f"Esquema: {config.catalog}.{config.schema} · escala {config.scale}")

## 1. Armar la colección

`collect_list(struct(...))` agrupa las filas de transacciones en un array por cliente: es la operación inversa a un join.

In [ ]:
import json

docs_table = qualified_table(config, "nosql_customer_documents")

tx_items = (spark.table(transactions).alias("t")
    .join(spark.table(products).alias("p"), "product_id")
    .select("t.customer_id", "t.amount", "t.is_fraud",
            F.struct(F.col("t.event_ts").cast("string").alias("event_ts"),   # primero: el array se ordena por fecha
                     "t.transaction_id",
                     F.col("t.amount").cast("double").alias("amount"),
                     "t.payment_channel", "t.device_id", "t.is_fraud",
                     F.struct("p.product_id", "p.category", F.col("p.price").cast("double").alias("price")).alias("product")
            ).alias("item")))

per_customer = (tx_items.groupBy("customer_id")
    .agg(F.sort_array(F.collect_list("item"), asc=False).alias("transactions"),
         F.struct(F.count("*").alias("tx_count"),
                  F.round(F.sum("amount").cast("double"), 2).alias("total_amount"),
                  F.sum("is_fraud").alias("fraud_transactions")).alias("stats")))

documents = (spark.table(customers).join(per_customer, "customer_id", "left")
    .select(F.col("customer_id").alias("_id"),
            F.struct("country", "segment", F.col("created_date").cast("string").alias("created_date"), "email").alias("profile"),
            "stats",
            F.coalesce("transactions", F.array()).alias("transactions")))

documents.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(docs_table)
spark.table(docs_table).write.mode("overwrite").json(f"{lab_path}/customer_documents")   # misma colección como JSON Lines
spark.table(docs_table).printSchema()
print(f"{spark.table(docs_table).count():,} documentos")

## 2. Mirar un documento

Mostramos el cliente 42 como JSON, recortando la lista a las tres transacciones más recientes.

In [ ]:
example_id = 42
raw = spark.table(docs_table).where(F.col("_id") == example_id).select(F.to_json(F.struct("*"))).first()[0]
doc = json.loads(raw)
total = len(doc["transactions"])
doc["transactions"] = doc["transactions"][:3] + ([f"... {total - 3} transacciones más"] if total > 3 else [])
print(json.dumps(doc, indent=2, ensure_ascii=False))

## 3. La misma pregunta, dos modelos

*"Mostrame el cliente 42 con todo su historial"*. En el modelo relacional hay que unir tres tablas y el dato del cliente se repite en cada fila. En el documental se lee **un solo documento**.

In [ ]:
relational = spark.sql(f"""
SELECT c.customer_id, c.country, c.segment, t.transaction_id, t.event_ts, t.amount, p.category
FROM {customers} c
JOIN {transactions} t ON t.customer_id = c.customer_id
JOIN {products} p ON p.product_id = t.product_id
WHERE c.customer_id = {example_id}
""")
document = spark.table(docs_table).where(F.col("_id") == example_id)

print(f"Relacional: 3 tablas, 2 joins, {relational.count()} filas devueltas")
print(f"Documental: 1 colección, 0 joins, {document.count()} documento devuelto")
display(document.select("_id", "profile", "stats", F.size("transactions").alias("n_transactions")))

## 4. Consultar dentro del documento

Spark SQL entiende la estructura anidada: notación con punto (`profile.country`) y funciones sobre arrays como `size`, `exists`, `transform` y `filter`, sin "aplanar" los datos.

In [ ]:
spark.table(docs_table).createOrReplaceTempView("customer_docs")
display(spark.sql("""
SELECT _id,
       profile.country,
       stats.tx_count,
       stats.total_amount,
       exists(transactions, t -> t.amount > 1750)                    AS tiene_compra_grande,
       size(array_distinct(transform(transactions, t -> t.device_id))) AS dispositivos,
       transactions[0].event_ts                                      AS ultima_compra
FROM customer_docs
WHERE profile.segment = 'premium'
ORDER BY stats.total_amount DESC
LIMIT 10
"""))

En cambio, una pregunta que **cruza** documentos (*"monto total por categoría"*) obliga a desarmar los arrays con `explode`. El modelo documental favorece el acceso por entidad; para analítica transversal hay que volver a filas.

In [ ]:
by_category = spark.sql("""
SELECT t.product.category AS category,
       count(*)           AS transactions,
       round(sum(t.amount), 2) AS total_amount
FROM customer_docs
LATERAL VIEW explode(transactions) exploded AS t
GROUP BY t.product.category
ORDER BY total_amount DESC
""")
display(by_category)

## 5. Esquema flexible: documentos con formas distintas

En una colección no todos los documentos necesitan los mismos campos. Escribimos cuatro documentos nuevos con formas distintas y dejamos que Spark infiera el esquema al leerlos (*schema-on-read*).

In [ ]:
flexible_docs = [
    {"_id": 900001, "profile": {"country": "AR", "segment": "premium", "email": "vip@example.test"},
     "loyalty": {"tier": "gold", "points": 1200}},
    {"_id": 900002, "profile": {"country": "UY", "segment": "new"},                         # sin email
     "phones": ["+598 0000 0000", "+598 1111 1111"]},
    {"_id": 900003, "profile": {"country": "BR", "segment": "regular", "email": None}},      # email null explícito
    {"_id": 900004, "profile": {"country": "CL", "segment": "regular", "email": "d@example.test"},
     "schema_version": 2, "preferences": {"newsletter": True}},
]
flex_path = f"{lab_path}/flexible_documents/docs.json"
dbutils.fs.put(flex_path, "\n".join(json.dumps(d) for d in flexible_docs), overwrite=True)

flexible = spark.read.json(flex_path)
flexible.printSchema()
display(flexible)

Spark construye un esquema que es la **unión** de todos los campos y completa con `null` lo que falta. Al hacerlo se pierde una diferencia: ¿el email **no vino** o vino **explícitamente en null**? Leyendo el texto original se puede distinguir.

In [ ]:
display(spark.read.text(flex_path).select(
    F.get_json_object("value", "$._id").alias("_id"),
    F.col("value").contains('"email"').alias("trae_campo_email"),
    F.get_json_object("value", "$.profile.email").alias("email")))

## 6. ¿Cuánto pesa cada documento?

Embeber tiene un costo: el documento crece con cada compra. MongoDB limita un documento a **16 MB**. Medimos el tamaño en bytes de cada documento.

In [ ]:
sizes = (spark.table(docs_table)
    .select(F.size("transactions").alias("n_transactions"),
            F.length(F.to_json(F.struct("*"))).alias("bytes"))
    .toPandas())

bytes_per_tx = (sizes["bytes"] / sizes["n_transactions"].clip(lower=1)).median()
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].hist(sizes["n_transactions"], bins=30, color="#2563eb")
axes[0].set_title("Transacciones embebidas por documento")
axes[0].set_xlabel("Transacciones")
axes[0].set_ylabel("Documentos")
axes[1].hist(sizes["bytes"] / 1024, bins=30, color="#9333ea")
axes[1].set_title("Tamaño del documento")
axes[1].set_xlabel("KB")
fig.tight_layout()
plt.show()
print(f"Documento más grande: {sizes['bytes'].max() / 1024:.1f} KB · ≈{bytes_per_tx:.0f} bytes por transacción")
print(f"A ese ritmo, un cliente llegaría a 16 MB con ≈{16 * 1024 * 1024 / bytes_per_tx:,.0f} transacciones")

## Tu turno

Usando **sólo** la vista `customer_docs` (sin `JOIN` ni `explode`), listá los clientes que tienen **al menos dos dispositivos distintos** y **al menos una transacción con `is_fraud = 1`**. Mostrá `_id`, país, cantidad de dispositivos y cantidad de transacciones fraudulentas (pista: `filter(transactions, t -> ...)`).

In [ ]:
# Tu solución